# Chapter 1: From Pixels to Sequences — Vision Transformer

*Build a Multimodal Model from Scratch*

---

> **Prerequisites**: You have read *Build a Large Language Model from Scratch*
> and understand Transformers, multi-head attention, residual connections,
> and layer normalization. Those concepts are not re-explained here.

---

**What you will build in this chapter:**

```
Image (32×32)  →  PatchEmbedding  →  [CLS | P1 | P2 | … | P16]  →  ViT Blocks  →  Classifier
```

By the end, you will have a working Vision Transformer that classifies
images with >95% accuracy on a synthetic dataset — and you will understand
every line of code that makes it work.

| Section | What you will learn |
|---------|---------------------|
| 1.1 | Why we split images into patches, not pixels |
| 1.2 | How `Conv2d` secretly does patch extraction + projection in one step |
| 1.3 | Why Transformers need positional embeddings |
| 1.4 | The role of the `[CLS]` token |
| 1.5 | The single difference between ViT and GPT attention |
| 1.6 | Assembling the full model |
| 1.7 | Training and evaluating on synthetic data |

In [ ]:
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# ── Consistent color palette used across all figures ──
C = dict(
    bg     = '#F8F9FA',  # light grey background
    dark   = '#2C3E50',  # near-black text
    blue   = '#4A90D9',  # primary blue
    orange = '#E67E22',  # accent orange
    green  = '#27AE60',  # correct / positive
    red    = '#E74C3C',  # wrong / negative
    purple = '#8E44AD',  # secondary
    grey   = '#95A5A6',  # muted
)

---
## 1.0  The Big Picture

Before diving into code, let's see the complete ViT pipeline in one diagram.
Every section in this chapter implements one box in this figure.

In [ ]:
def draw_vit_overview():
    """Full ViT pipeline: image → patches → tokens → transformer → classifier."""
    fig, ax = plt.subplots(figsize=(14, 4))
    fig.patch.set_facecolor(C['bg'])
    ax.set_facecolor(C['bg'])
    ax.axis('off')

    # ── Draw boxes ──────────────────────────────────────────────────────────
    boxes = [
        (0.04, 0.20, 0.10, 0.60, C['blue'],   'Input\nImage\n(H, W, C)',   'white'),
        (0.19, 0.15, 0.13, 0.70, C['orange'], 'Patch\nEmbedding\n(N, D)',  'white'),
        (0.37, 0.15, 0.13, 0.70, C['purple'], '[CLS] + Pos\nEmbedding\n(N+1, D)', 'white'),
        (0.55, 0.15, 0.13, 0.70, C['dark'],   'Transformer\nBlocks\n\u00d7 L',  'white'),
        (0.73, 0.15, 0.13, 0.70, C['dark'],   'CLS Token\nOutput\n(D,)',   'white'),
        (0.89, 0.20, 0.09, 0.60, C['green'],  'Class\nLogits\n(K,)',       'white'),
    ]
    for (x, y, w, h, color, label, fc) in boxes:
        rect = mpatches.FancyBboxPatch(
            (x, y), w, h, boxstyle='round,pad=0.01',
            facecolor=color, edgecolor='white', linewidth=1.5,
            transform=ax.transAxes, clip_on=False
        )
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha='center', va='center',
                color=fc, fontsize=9.5, fontweight='bold',
                transform=ax.transAxes)

    # ── Arrows between boxes ────────────────────────────────────────────────
    arrow_xs = [0.145, 0.330, 0.510, 0.690, 0.870]
    for x in arrow_xs:
        ax.annotate('', xy=(x + 0.04, 0.50), xytext=(x, 0.50),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color=C['dark'],
                                   lw=2.0, mutation_scale=18))

    # ── Section labels under each box ──────────────────────────────────────
    labels_below = [
        (0.09,  '\u00a7 1.1'),
        (0.255, '\u00a7 1.2'),
        (0.435, '\u00a7 1.3-1.4'),
        (0.615, '\u00a7 1.5-1.6'),
        (0.795, '\u00a7 1.6'),
        (0.935, '\u00a7 1.7'),
    ]
    for (x, txt) in labels_below:
        ax.text(x, 0.08, txt, ha='center', va='center',
                color=C['grey'], fontsize=8.5, transform=ax.transAxes)

    ax.set_title('Vision Transformer (ViT) — Complete Pipeline',
                 fontsize=13, color=C['dark'], pad=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

draw_vit_overview()

---
## 1.1  Why Images Need Special Treatment

> **Analogy**: Imagine describing a painting to someone by reading out every
> pixel's RGB value, one by one.  You would spend hours on the painting's
> texture before you could say anything meaningful about the subject.
> Humans recognize objects in *regions*, not in isolated pixels.

The same problem hits Transformers:

```
224 x 224 x 3  =  150,528 tokens per image
```

Attention is O(N²) in sequence length, so a 150K-token sequence is
computationally infeasible.  The solution: **group nearby pixels into patches**.

```
224 x 224 image,  patch size 16:
  (224 / 16) x (224 / 16)  =  14 x 14  =  196 patches
```

196 tokens instead of 150,528 — a **768x reduction** — while keeping enough
spatial detail for vision tasks.

In [ ]:
def draw_patch_splitting():
    """Visualise image -> patches -> flat vectors -> projected embeddings."""
    fig = plt.figure(figsize=(14, 5))
    fig.patch.set_facecolor(C['bg'])
    gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.08)

    # Panel 1 — Original image with patch grid overlay
    ax1 = fig.add_subplot(gs[0])
    ax1.set_facecolor(C['bg'])
    img_size, patch_size = 48, 16
    n = img_size // patch_size   # 3 patches per side
    colors_2d = [
        ['#e74c3c', '#3498db', '#2ecc71'],
        ['#f39c12', '#9b59b6', '#1abc9c'],
        ['#e67e22', '#34495e', '#16a085'],
    ]
    for row in range(n):
        for col in range(n):
            rect = mpatches.Rectangle(
                (col * patch_size + 1, row * patch_size + 1),
                patch_size - 2, patch_size - 2,
                facecolor=colors_2d[row][col], edgecolor='white', linewidth=1.5
            )
            ax1.add_patch(rect)
            ax1.text(col * patch_size + patch_size/2,
                     row * patch_size + patch_size/2,
                     f'P{row*n+col+1}', ha='center', va='center',
                     color='white', fontsize=10, fontweight='bold')
    ax1.set_xlim(0, img_size)
    ax1.set_ylim(img_size, 0)
    ax1.set_title(f'Image ({img_size}x{img_size})\n9 patches', color=C['dark'],
                  fontsize=10, fontweight='bold')
    ax1.axis('off')

    # Arrow
    for ax_from, ax_to in [(ax1, None)]:
        pass  # arrows added with fig.transFigure below

    # Panel 2 — Flattened patch vectors
    ax2 = fig.add_subplot(gs[1])
    ax2.set_facecolor(C['bg'])
    flat_colors = [c for row in colors_2d for c in row]
    for i, fc in enumerate(flat_colors):
        bar_h = 0.7 / len(flat_colors)
        rect = mpatches.Rectangle(
            (0.1, 0.95 - (i + 1) * bar_h * 1.1),
            0.8, bar_h * 0.9,
            facecolor=fc, edgecolor='white', linewidth=0.8,
            transform=ax2.transAxes, clip_on=False
        )
        ax2.add_patch(rect)
        ax2.text(0.50, 0.95 - (i + 0.5) * bar_h * 1.1, f'P{i+1}  [C*P*P]',
                 ha='center', va='center', color='white', fontsize=7.5,
                 fontweight='bold', transform=ax2.transAxes)
    ax2.set_title('Flatten each patch\n(C x P x P) vector', color=C['dark'],
                  fontsize=10, fontweight='bold')
    ax2.axis('off')

    # Panel 3 — Linear projection
    ax3 = fig.add_subplot(gs[2])
    ax3.set_facecolor(C['bg'])
    ax3.text(0.5, 0.55, 'Linear\nProjection\n\nW  (C·P²  x  D)',
             ha='center', va='center', color='white', fontsize=10,
             fontweight='bold', transform=ax3.transAxes,
             bbox=dict(boxstyle='round,pad=0.5', facecolor=C['orange'],
                       edgecolor='white', linewidth=2))
    ax3.set_title('Learnable weight\nmatrix', color=C['dark'],
                  fontsize=10, fontweight='bold')
    ax3.axis('off')

    # Panel 4 — Embedded tokens
    ax4 = fig.add_subplot(gs[3])
    ax4.set_facecolor(C['bg'])
    for i, fc in enumerate(flat_colors):
        bar_h = 0.7 / len(flat_colors)
        rect = mpatches.Rectangle(
            (0.1, 0.95 - (i + 1) * bar_h * 1.1),
            0.8, bar_h * 0.9,
            facecolor=C['blue'], edgecolor='white', linewidth=0.8,
            transform=ax4.transAxes, clip_on=False
        )
        ax4.add_patch(rect)
        ax4.text(0.50, 0.95 - (i + 0.5) * bar_h * 1.1, f'e{i+1}  [D]',
                 ha='center', va='center', color='white', fontsize=7.5,
                 fontweight='bold', transform=ax4.transAxes)
    ax4.set_title('Patch embeddings\nshape: (N, D)', color=C['dark'],
                  fontsize=10, fontweight='bold')
    ax4.axis('off')

    # Arrows between panels
    for x_from, x_to in [(0.26, 0.30), (0.50, 0.54), (0.73, 0.77)]:
        fig.annotate('', xy=(x_to, 0.50), xytext=(x_from, 0.50),
                     xycoords='figure fraction', textcoords='figure fraction',
                     arrowprops=dict(arrowstyle='->', color=C['dark'],
                                    lw=2.5, mutation_scale=20))

    fig.suptitle('Patch Embedding: Image \u2192 Token Sequence',
                 fontsize=13, color=C['dark'], fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

draw_patch_splitting()

---
## 1.2  Patch Embedding: From Scratch

> **Key idea**: Every patch is a small rectangular region of the image.
> We flatten it into a 1-D vector, then multiply by a learned weight matrix
> to get a D-dimensional embedding.  All patches share the same weight matrix.

We build two versions to make sure you truly understand what is happening:

1. **Naive** — explicit Python loop, crystal-clear but slow
2. **Fast** — a single `Conv2d` call that does *exactly the same math*

### Version 1: Naive Loop

In [ ]:
class PatchEmbeddingNaive(nn.Module):
    """
    Pedagogical implementation: explicit loop over every patch.
    Slow but 100% transparent.
    """
    def __init__(self, patch_size: int, in_channels: int, embed_dim: int):
        super().__init__()
        self.patch_size = patch_size
        # One shared linear projection applied to every patch
        # Input size: C * P * P  (flattened patch pixels)
        self.linear = nn.Linear(in_channels * patch_size * patch_size, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape      # e.g. (8, 3, 32, 32)
        P = self.patch_size        # e.g. 8
        n_h = H // P               # patches per column
        n_w = W // P               # patches per row

        patches = []
        for i in range(n_h):
            for j in range(n_w):
                # Slice one patch: (B, C, P, P)
                patch = x[:, :, i*P:(i+1)*P, j*P:(j+1)*P]
                # Flatten pixels: (B, C*P*P)
                patch = patch.reshape(B, -1)
                # Project to embed_dim: (B, D)
                patches.append(self.linear(patch))

        # Stack into sequence: (B, N_patches, D)
        return torch.stack(patches, dim=1)


# Quick sanity check: verify output shape
x_test  = torch.randn(2, 3, 32, 32)   # batch=2, RGB, 32x32 image
naive   = PatchEmbeddingNaive(patch_size=8, in_channels=3, embed_dim=64)
out     = naive(x_test)
# Expected: (2, 16, 64) — 2 images, 4x4=16 patches, 64-dim embeddings
print(f'Input:  {tuple(x_test.shape)}')
print(f'Output: {tuple(out.shape)}   <- (batch, n_patches, embed_dim)')

### Version 2: The Conv2d Trick

> **Key insight**: A convolution with `kernel_size = stride = patch_size`
> slides a window across the image *without overlap*.  Each application of
> the kernel flattens one patch and applies a linear projection — which is
> **mathematically identical** to the loop above, but fully vectorized.

```
Conv2d(in_channels=C, out_channels=D,
       kernel_size=P, stride=P)

  kernel_size = P  ->  each kernel covers exactly one P x P patch
  stride      = P  ->  no overlap between patches
  out_channels = D  ->  D output values per patch = projection to D dims
```

After the conv: shape is `(B, D, H/P, W/P)`.
We just need to flatten the spatial dimensions: `flatten(2).transpose(1, 2)`
gives `(B, N, D)` — exactly what we want.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Production-quality implementation using Conv2d.
    Mathematically identical to PatchEmbeddingNaive but fully vectorized.
    """
    def __init__(self, img_size: int = 224, patch_size: int = 16,
                 in_channels: int = 3, embed_dim: int = 768):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        # kernel_size=stride=patch_size => non-overlapping patch extraction + projection
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x:    (B, C, H, W)
        # proj: (B, D, H/P, W/P)  after conv
        # flatten(2): (B, D, N)   collapse spatial dims
        # transpose:  (B, N, D)   swap to sequence-first
        return self.proj(x).flatten(2).transpose(1, 2)


# ── Prove that both implementations are numerically equivalent ────────────────
torch.manual_seed(0)
x_test = torch.randn(2, 3, 32, 32)
P, D   = 8, 64

naive = PatchEmbeddingNaive(P, 3, D)
fast  = PatchEmbedding(img_size=32, patch_size=P, in_channels=3, embed_dim=D)

# Copy weights from Conv2d into the Linear layer so both use identical params
# Conv2d weight shape: (D, C, P, P)  ->  reshape to  (D, C*P*P)  for Linear
with torch.no_grad():
    naive.linear.weight.copy_(fast.proj.weight.view(D, -1))
    naive.linear.bias.copy_(fast.proj.bias)

out_naive = naive(x_test)
out_fast  = fast(x_test)
max_diff  = (out_naive - out_fast).abs().max().item()

print(f'Naive output shape : {tuple(out_naive.shape)}')
print(f'Fast  output shape : {tuple(out_fast.shape)}')
print(f'Max absolute diff  : {max_diff:.2e}   <- should be near zero')

---
## 1.3  Positional Embedding

> **Analogy**: Imagine cutting a newspaper article into sentences and shuffling
> them randomly.  Without knowing the order, the Transformer cannot tell which
> sentence was the opening paragraph.  Positional embeddings are the
> page numbers that restore the order.

Transformers are **permutation-invariant** by design: the attention mechanism
treats all input tokens identically regardless of their position in the sequence.
This is fine for sets, but images are grids — the top-left patch and the
bottom-right patch carry different spatial meaning.

The following figure shows what happens when we shuffle patches:

In [ ]:
def show_shuffle_effect():
    """Side-by-side: original vs patch-shuffled image."""
    torch.manual_seed(7)
    img = torch.zeros(1, 3, 32, 32)
    img[0, 0, :16, :16] = 0.9    # top-left: red
    img[0, 2, 16:, 16:] = 0.9    # bottom-right: blue
    img[0, 1, :16, 16:] = 0.9    # top-right: green
    img[0, 0, 16:, :16] = 0.5    # bottom-left: dark-red
    img[0, 1, 16:, :16] = 0.5

    # Extract 8x8 patches, shuffle them, reassemble
    P = 8
    patches = img.unfold(2, P, P).unfold(3, P, P)  # (1, 3, 4, 4, P, P)
    B, C, nh, nw, _, _ = patches.shape
    patches = patches.permute(0, 2, 3, 1, 4, 5).reshape(-1, C, P, P)
    idx = torch.randperm(patches.shape[0])
    shuffled = patches[idx].reshape(nh, nw, C, P, P)
    shuffled = shuffled.permute(2, 0, 3, 1, 4).reshape(C, nh*P, nw*P).unsqueeze(0)

    def to_np(t):
        return t.squeeze(0).permute(1, 2, 0).clamp(0, 1).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
    fig.patch.set_facecolor(C['bg'])

    axes[0].imshow(to_np(img))
    axes[0].set_title('Original image', color=C['dark'], fontweight='bold')

    axes[1].imshow(to_np(shuffled))
    axes[1].set_title('After shuffling patches\n(Transformer sees this as the same!)',
                      color=C['red'], fontweight='bold')

    # Draw patch grid overlay on shuffled image
    for i in range(nh + 1):
        axes[1].axhline(i * P - 0.5, color='white', lw=0.8)
    for j in range(nw + 1):
        axes[1].axvline(j * P - 0.5, color='white', lw=0.8)

    # Positional embedding fixes it
    axes[2].imshow(to_np(img))
    for i in range(nh + 1):
        axes[2].axhline(i * P - 0.5, color='white', lw=0.8)
    for j in range(nw + 1):
        axes[2].axvline(j * P - 0.5, color='white', lw=0.8)
    for i in range(nh):
        for j in range(nw):
            axes[2].text(j*P + P/2, i*P + P/2, f'pos\n{i*nw+j}',
                         ha='center', va='center',
                         color='white', fontsize=7, fontweight='bold')
    axes[2].set_title('Positional embedding\n(restores spatial order)',
                      color=C['green'], fontweight='bold')

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_shuffle_effect()

ViT uses **learnable** positional embeddings — a parameter tensor that is
trained end-to-end along with the rest of the model.  Unlike the fixed
sinusoidal embeddings in the original Transformer paper, learned embeddings
can adapt to the specific spatial patterns of the training distribution.

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    Learnable positional embedding for ViT.

    Shape: (1, n_patches + 1, embed_dim)
    The +1 is for the CLS token (added in the next section).
    The leading 1 broadcasts over the batch dimension automatically.
    """
    def __init__(self, n_patches: int, embed_dim: int):
        super().__init__()
        # Initialise near zero; will be learned during training
        self.pos_embed = nn.Parameter(
            torch.zeros(1, n_patches + 1, embed_dim)
        )
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x:           (B, n_patches+1, embed_dim)
        # pos_embed:   (1, n_patches+1, embed_dim)  <- broadcast over batch
        return x + self.pos_embed

---
## 1.4  The [CLS] Token

> **Analogy**: Think of a meeting where N employees each give a status update.
> The manager listens to everyone via attention and then writes a summary in
> their notepad.  The `[CLS]` token is the manager's notepad — it attends to
> all patches and accumulates a global image representation.

The Transformer outputs one vector per input token.  We need a *single* vector
to represent the whole image.  Two common strategies:

| Strategy | How | Used by |
|----------|-----|--------|
| **[CLS] token** | Prepend a learnable token; read its output | ViT, BERT |
| **Average pooling** | Average all patch token outputs | DeiT, some ViT variants |

We use the CLS token (original ViT design).  In code, it is a single
learnable vector of shape `(1, 1, D)` that is prepended to the patch sequence
before the Transformer blocks:

In [ ]:
# Illustration: how CLS token is prepended
B, N, D = 4, 16, 64   # batch=4, patches=16, embed_dim=64
patch_tokens = torch.randn(B, N, D)               # patch embeddings

cls_token = nn.Parameter(torch.zeros(1, 1, D))    # learnable; shape (1,1,D)
cls_expanded = cls_token.expand(B, -1, -1)        # (B, 1, D)  -- one per image

# Concatenate: CLS token first, then all patch tokens
tokens = torch.cat([cls_expanded, patch_tokens], dim=1)  # (B, N+1, D)

print(f'Patch tokens : {tuple(patch_tokens.shape)}')
print(f'CLS token    : {tuple(cls_expanded.shape)}')
print(f'After concat : {tuple(tokens.shape)}   <- [CLS, P1, P2, ..., P16]')
print()
print('After Transformer blocks, we read only tokens[:, 0, :]  (the CLS output)')

---
## 1.5  ViT Attention — One Change From GPT

> **Key point**: ViT and GPT use *identical* multi-head attention code.
> The only difference is that ViT does NOT add a causal mask.

| | GPT (text generation) | ViT (image encoding) |
|--|--|--|
| Attention type | **Causal** (each token sees only past tokens) | **Bidirectional** (every patch sees all patches) |
| Mask | Upper triangle set to -inf | No mask |
| Why | Language is sequential: future words not yet generated | Images are spatial: every patch can inform every other |

The figure below shows the attention patterns side-by-side:

In [ ]:
def draw_attention_masks():
    """Causal (GPT) vs bidirectional (ViT) attention patterns."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor(C['bg'])

    T      = 6
    labels = ['CLS', 'P1', 'P2', 'P3', 'P4', 'P5']

    for ax, title, causal, accent in [
        (axes[0], 'ViT  --  Bidirectional Attention\n(every patch attends to all patches)',
         False, C['blue']),
        (axes[1], 'GPT  --  Causal Attention\n(each token attends only to past tokens)',
         True,  C['red']),
    ]:
        ax.set_facecolor(C['bg'])

        for i in range(T):
            for j in range(T):
                if causal and j > i:
                    color = '#E8E8E8'   # masked out
                    txt   = 'X'
                    tc    = '#AAAAAA'
                else:
                    color = accent
                    txt   = 'O'
                    tc    = 'white'
                rect = mpatches.FancyBboxPatch(
                    (j + 0.05, T - i - 1 + 0.05), 0.9, 0.9,
                    boxstyle='round,pad=0.02',
                    facecolor=color, edgecolor='white', linewidth=1,
                )
                ax.add_patch(rect)
                ax.text(j + 0.50, T - i - 0.50, txt, ha='center', va='center',
                        color=tc, fontsize=13, fontweight='bold')

        ax.set_xlim(0, T)
        ax.set_ylim(0, T)
        ax.set_xticks([x + 0.5 for x in range(T)])
        ax.set_yticks([y + 0.5 for y in range(T)])
        ax.set_xticklabels(labels, fontsize=9, color=C['dark'])
        ax.set_yticklabels(labels[::-1], fontsize=9, color=C['dark'])
        ax.set_xlabel('Key (what is attended TO)', color=C['dark'])
        ax.set_ylabel('Query (who is attending)', color=C['dark'])
        ax.set_title(title, color=C['dark'], fontsize=10, fontweight='bold', pad=10)
        ax.tick_params(length=0)

        # Legend
        handles = [
            mpatches.Patch(facecolor=accent,    label='Can attend (O)'),
            mpatches.Patch(facecolor='#E8E8E8', label='Masked out (X)'),
        ]
        ax.legend(handles=handles, loc='lower right', fontsize=8,
                  framealpha=0.8)

    plt.tight_layout(pad=2.0)
    plt.show()

draw_attention_masks()

In code, the only change from GPT is removing the causal mask:

```python
# GPT -- add this mask to block future tokens:
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))

# ViT -- simply DON'T add the mask.
# That's the entire difference.
```

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Bidirectional multi-head self-attention (no causal mask)."""

    def __init__(self, embed_dim: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert embed_dim % n_heads == 0, 'embed_dim must be divisible by n_heads'
        self.n_heads  = n_heads
        self.head_dim = embed_dim // n_heads   # each head works in smaller space
        self.scale    = self.head_dim ** -0.5  # 1/sqrt(head_dim) prevents vanishing gradients

        # Single matrix projects input to Q, K, V simultaneously
        self.qkv      = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.drop     = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape   # batch, sequence length (N_patches+1), embed_dim

        # Project and split into Q, K, V
        qkv = self.qkv(x)                         # (B, N, 3*C)
        qkv = qkv.reshape(B, N, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)          # (3, B, H, N, head_dim)
        q, k, v = qkv.unbind(0)                    # each: (B, H, N, head_dim)

        # Scaled dot-product attention (no causal mask for ViT)
        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, H, N, N)
        attn = F.softmax(attn, dim=-1)
        attn = self.drop(attn)

        # Weighted sum of values, then merge heads
        out = (attn @ v)                           # (B, H, N, head_dim)
        out = out.transpose(1, 2).reshape(B, N, C) # (B, N, C)
        return self.out_proj(out)                  # (B, N, C)

---
## 1.6  Assembling the Full ViT

Each Transformer block uses Pre-LayerNorm (normalize *before* the sub-layer),
which is more stable than the original Post-LN formulation:

```
x  =  x  +  Attention( LayerNorm(x) )     # residual + attention
x  =  x  +  FFN(       LayerNorm(x) )     # residual + feed-forward
```

This is **identical** to GPT's block.  Everything above it (patch embedding,
CLS token, bidirectional attention) is what distinguishes ViT.

In [ ]:
class ViTBlock(nn.Module):
    """Single ViT Transformer block: Pre-LN + bidirectional attention + FFN."""

    def __init__(self, embed_dim: int, n_heads: int,
                 mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        # FFN hidden dim is typically 4x the embedding dim
        hidden = int(embed_dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),             # smoother than ReLU; standard in modern Transformers
            nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))   # attention sub-layer with residual
        x = x + self.ffn(self.norm2(x))    # FFN sub-layer with residual
        return x


class VisionTransformer(nn.Module):
    """
    Full Vision Transformer encoder.

    Two output modes (controlled by return_all_tokens):
      False (default)  ->  CLS token only:  (B, embed_dim)
      True             ->  all patch tokens: (B, N_patches, embed_dim)
                           Used by the VLM in Chapter 3.
    """

    def __init__(self, img_size: int = 224, patch_size: int = 16,
                 in_channels: int = 3, embed_dim: int = 768,
                 depth: int = 12, n_heads: int = 12,
                 mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches        = self.patch_embed.n_patches

        self.cls_token  = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed  = PositionalEmbedding(n_patches, embed_dim)
        self.blocks     = nn.ModuleList(
            [ViTBlock(embed_dim, n_heads, mlp_ratio, dropout) for _ in range(depth)]
        )
        self.norm       = nn.LayerNorm(embed_dim)   # final normalisation

        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x: torch.Tensor,
                return_all_tokens: bool = False) -> torch.Tensor:
        B = x.shape[0]

        # 1. Split image into patch tokens
        tokens = self.patch_embed(x)                                # (B, N, D)

        # 2. Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)                     # (B, 1, D)
        tokens = torch.cat([cls, tokens], dim=1)                    # (B, N+1, D)

        # 3. Add positional embedding
        tokens = self.pos_embed(tokens)                             # (B, N+1, D)

        # 4. Pass through Transformer blocks
        for block in self.blocks:
            tokens = block(tokens)                                  # (B, N+1, D)

        tokens = self.norm(tokens)                                  # (B, N+1, D)

        if return_all_tokens:
            return tokens[:, 1:, :]   # drop CLS, return (B, N, D) patch tokens
        return tokens[:, 0, :]        # CLS token only: (B, D)

---
## 1.7  Training Demo: 4-Class Image Classification

We now verify that our ViT implementation actually learns.  The dataset
contains four simple visual patterns — distinct enough that a working model
should reach near-100% accuracy.

| Label | Pattern | How it looks |
|-------|---------|-------------|
| 0 | Solid red | Entire image filled with red |
| 1 | Solid blue | Entire image filled with blue |
| 2 | Vertical stripes | Alternating red/blue columns |
| 3 | Horizontal stripes | Alternating red/blue rows |

In [ ]:
class SyntheticImageDataset(Dataset):
    """4-class geometric image dataset for ViT training validation."""

    def __init__(self, n_per_class: int = 200, img_size: int = 32, seed: int = 42):
        torch.manual_seed(seed)
        self.data = []
        for label in range(4):
            for _ in range(n_per_class):
                img = torch.zeros(3, img_size, img_size)
                noise = torch.randn_like(img) * 0.05   # tiny noise keeps it interesting
                if label == 0:    # solid red
                    img[0] = 0.9
                elif label == 1:  # solid blue
                    img[2] = 0.9
                elif label == 2:  # vertical stripes
                    img[0, :, ::2]  = 0.9   # red columns
                    img[2, :, 1::2] = 0.9   # blue columns
                else:             # horizontal stripes
                    img[0, ::2, :]  = 0.9   # red rows
                    img[2, 1::2, :] = 0.9   # blue rows
                self.data.append((img + noise, label))

    def __len__(self):  return len(self.data)
    def __getitem__(self, i): return self.data[i]


def visualize_dataset_samples():
    """Show 4 sample images (one per class) to verify the dataset."""
    ds = SyntheticImageDataset(n_per_class=1)
    class_names = ['Solid Red', 'Solid Blue', 'Vertical Stripes', 'Horizontal Stripes']

    fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
    fig.patch.set_facecolor(C['bg'])
    for i, (img, label) in enumerate(ds):
        axes[i].imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
        axes[i].set_title(f'Class {label}\n{class_names[label]}',
                          color=C['dark'], fontsize=9, fontweight='bold')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

visualize_dataset_samples()

In [ ]:
class ViTClassifier(nn.Module):
    """ViT encoder + linear classification head."""

    def __init__(self, n_classes: int, **vit_kwargs):
        super().__init__()
        self.vit  = VisionTransformer(**vit_kwargs)
        embed_dim = vit_kwargs.get('embed_dim', 768)
        self.head = nn.Linear(embed_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        cls_out = self.vit(x)          # (B, embed_dim)  CLS token output
        return self.head(cls_out)      # (B, n_classes)  class logits


# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS     = 30
BATCH_SIZE = 64
LR         = 3e-4

torch.manual_seed(42)
dataset = SyntheticImageDataset(n_per_class=200, img_size=32)

# 80/20 train/test split
n_train = int(0.8 * len(dataset))
train_ds, test_ds = torch.utils.data.random_split(
    dataset, [n_train, len(dataset) - n_train]
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

model = ViTClassifier(
    n_classes  = 4,
    img_size   = 32,
    patch_size = 8,    # 4x4 = 16 patches per image
    in_channels= 3,
    embed_dim  = 128,
    depth      = 4,
    n_heads    = 4,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


# ── Training loop ─────────────────────────────────────────────────────────────
train_losses, train_accs, test_accs = [], [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    avg_loss  = epoch_loss / len(train_loader)
    train_acc = evaluate(model, train_loader)
    test_acc  = evaluate(model, test_loader)
    train_losses.append(avg_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'loss={avg_loss:.4f}  '
              f'train_acc={train_acc:.1%}  '
              f'test_acc={test_acc:.1%}')

In [ ]:
def plot_training_results(losses, train_accs, test_accs):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor(C['bg'])

    epochs = range(1, len(losses) + 1)

    # Loss curve
    axes[0].set_facecolor(C['bg'])
    axes[0].plot(epochs, losses, color=C['blue'], lw=2.5, label='Train loss')
    axes[0].axhline(math.log(4), color=C['red'], linestyle='--', alpha=0.7,
                    label=f'Random baseline (ln 4 = {math.log(4):.2f})')
    axes[0].set_xlabel('Epoch', color=C['dark'])
    axes[0].set_ylabel('Cross-entropy loss', color=C['dark'])
    axes[0].set_title('Training Loss', color=C['dark'], fontweight='bold')
    axes[0].legend()
    axes[0].spines[['top', 'right']].set_visible(False)

    # Accuracy curves
    axes[1].set_facecolor(C['bg'])
    axes[1].plot(epochs, [a * 100 for a in train_accs], color=C['blue'],
                 lw=2.5, label='Train accuracy')
    axes[1].plot(epochs, [a * 100 for a in test_accs],  color=C['orange'],
                 lw=2.5, linestyle='--', label='Test accuracy')
    axes[1].axhline(25, color=C['red'], linestyle=':', alpha=0.7, label='Random (25%)')
    axes[1].set_xlabel('Epoch', color=C['dark'])
    axes[1].set_ylabel('Accuracy (%)', color=C['dark'])
    axes[1].set_ylim(0, 105)
    axes[1].set_title('Classification Accuracy', color=C['dark'], fontweight='bold')
    axes[1].legend()
    axes[1].spines[['top', 'right']].set_visible(False)

    for ax in axes:
        ax.tick_params(colors=C['dark'])

    plt.tight_layout()
    plt.show()

plot_training_results(train_losses, train_accs, test_accs)

In [ ]:
def visualize_predictions(model, dataset, n_samples: int = 8):
    """Show sample images with predicted vs true labels (green=correct, red=wrong)."""
    class_names = ['Solid Red', 'Solid Blue', 'V-Stripes', 'H-Stripes']
    model.eval()

    indices = random.sample(range(len(dataset)), n_samples)
    imgs    = torch.stack([dataset[i][0] for i in indices]).to(DEVICE)
    labels  = [dataset[i][1] for i in indices]

    with torch.no_grad():
        preds = model(imgs).argmax(dim=1).cpu().tolist()

    fig, axes = plt.subplots(2, n_samples // 2, figsize=(13, 5))
    fig.patch.set_facecolor(C['bg'])
    axes = axes.flatten()

    for k, ax in enumerate(axes):
        img   = imgs[k].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
        true  = labels[k]
        pred  = preds[k]
        color = C['green'] if pred == true else C['red']
        status= 'OK' if pred == true else 'WRONG'

        ax.imshow(img)
        ax.set_title(
            f'True: {class_names[true]}\nPred: {class_names[pred]}  [{status}]',
            color=color, fontsize=8, fontweight='bold'
        )
        # Coloured border
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    plt.suptitle('Sample Predictions (green = correct, red = wrong)',
                 color=C['dark'], fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_predictions(model, test_ds, n_samples=8)

---
## 1.8  Chapter Summary

You have built a complete Vision Transformer from scratch.  Here is what each
component does and why it is designed that way:

| Component | What it does | Key design choice |
|-----------|-------------|-------------------|
| `PatchEmbedding` | Splits image into N patches, projects each to D dims | `Conv2d(stride=P)` is equivalent to loop + Linear, but vectorized |
| `PositionalEmbedding` | Adds spatial position information to each token | Learnable parameters (not fixed sinusoidal) |
| `[CLS] token` | Accumulates global image representation | Prepended; its final output is used for classification |
| `MultiHeadSelfAttention` | Every patch attends to every other patch | **No causal mask** — this is the only code difference from GPT |
| `ViTBlock` | Attention + FFN with Pre-LayerNorm | Pre-LN is more stable than Post-LN |
| `VisionTransformer` | Orchestrates all components | `return_all_tokens=True` enables VLM use in Chapter 3 |

---

**What comes next?**

In Chapter 2, we will train two encoders — one for images, one for text — to
share a common embedding space.  This is CLIP: the model that enables
zero-shot image classification and is the foundation for every modern VLM.